# LeetCode 447 — Number of Boomerangs bằng Python

Link đề: https://leetcode.com/problems/number-of-boomerangs/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Math

Skeleton LeetCode:

```python
class Solution:
    def numberOfBoomerangs(self, points: List[List[int]]) -> int:
        ...
```

Ghi chú: đề và ví dụ được viết lại theo hiểu biết về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code (đối chiếu với vét cạn mọi bộ ba có thứ tự), nhưng nên đối chiếu lại đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số **bộ ba có thứ tự** `(i, j, k)` các điểm phân biệt sao cho khoảng cách từ `i` đến `j` **bằng** khoảng cách từ `i` đến `k` (thứ tự quan trọng).
2. Nắm các công cụ Python: duyệt hai tầng `for`, **bình phương khoảng cách** `dx*dx + dy*dy` (tránh căn bậc hai và số thực), `dict`/`collections.Counter` đếm theo khóa, `sorted` rồi đếm đoạn bằng nhau, và công thức hoán vị `x * (x - 1)`.
3. Đi từ **ba vòng lặp `O(n³)`** → **cố định tâm `i`, đếm khoảng cách bằng bảng băm, cộng `x(x-1)`** → **cộng dồn ngay khi gặp (`+= 2 * cnt[d]`)** → **sắp xếp khoảng cách rồi đếm đoạn bằng nhau** → **mỗi cặp điểm chỉ tính một lần, cập nhật bảng cho cả hai đầu**.
4. Hiểu mấu chốt: bộ ba `(i, j, k)` chỉ phụ thuộc vào **điểm giữa `i`** và **nhóm các điểm cùng khoảng cách tới `i`**; nếu nhóm có `x` điểm thì chọn `(j, k)` có thứ tự: `x * (x - 1)` cách.
5. Tránh các lỗi kinh điển: dùng tổ hợp `C(x, 2)` thay vì hoán vị (quên thứ tự), dùng chung một bảng đếm cho mọi tâm, đếm theo `(dx, dy)` thay vì khoảng cách, và dùng số thực (`sqrt`) để so sánh bằng.

## 2. Đề bài & ví dụ

Cho `n` điểm **đôi một khác nhau** trên mặt phẳng, `points[i] = [x_i, y_i]`. Một **boomerang** là một bộ ba `(i, j, k)` sao cho khoảng cách giữa `i` và `j` **bằng** khoảng cách giữa `i` và `k` (**thứ tự trong bộ ba có ý nghĩa**: `(i, j, k)` và `(i, k, j)` được tính là hai boomerang khác nhau).

Trả về **số boomerang**.

**Ví dụ 1**

- Input: `points = [[0,0],[1,0],[2,0]]`
- Output: `2`
- Giải thích: hai boomerang là `[[1,0],[0,0],[2,0]]` và `[[1,0],[2,0],[0,0]]` (tâm là `[1,0]`).

**Ví dụ 2**

- Input: `points = [[1,1],[2,2],[3,3]]`
- Output: `2`

**Ví dụ 3**

- Input: `points = [[1,1]]`
- Output: `0`

**Ràng buộc:**

- `n == points.length`, `1 <= n <= 500`
- `points[i].length == 2`
- `-10^4 <= x_i, y_i <= 10^4`
- Mọi điểm đều **khác nhau**.

In [1]:
from typing import List
from collections import Counter, defaultdict
from itertools import permutations
import random
import time
import math

examples = [
    ([[0, 0], [1, 0], [2, 0]], 2),
    ([[1, 1], [2, 2], [3, 3]], 2),
    ([[1, 1]], 0),
]
for pts, expected in examples:
    print(pts, "-> kỳ vọng", expected)

[[0, 0], [1, 0], [2, 0]] -> kỳ vọng 2
[[1, 1], [2, 2], [3, 3]] -> kỳ vọng 2
[[1, 1]] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Khoảng cách và vì sao dùng **bình phương** khoảng cách

Khoảng cách giữa `(x1, y1)` và `(x2, y2)` là `sqrt(dx² + dy²)` với `dx = x1 - x2`, `dy = y1 - y2`. Hai khoảng cách bằng nhau khi và chỉ khi **bình phương** của chúng bằng nhau (cả hai không âm). Bình phương `dx*dx + dy*dy` là **số nguyên**: so sánh chính xác, nhanh, không sai số. Với số thực (`sqrt`), kết quả có thể bị làm tròn (ví dụ `math.sqrt(2) ** 2 != 2`).

In [2]:
def d2(p, q):
    dx, dy = p[0] - q[0], p[1] - q[1]
    return dx * dx + dy * dy


print(d2([0, 0], [3, 4]))                  # 25 (khoảng cách thật là 5)
print(d2([1, 0], [0, 0]), d2([1, 0], [2, 0]))   # 1 1: hai điểm cùng cách [1,0] một khoảng 1
print(math.sqrt(2) ** 2 == 2, math.sqrt(2) ** 2)  # False 2.0000000000000004

25
1 1
False 2.0000000000000004


### 3.2. Hoán vị `x * (x - 1)` và tổ hợp `x * (x - 1) / 2`

Nếu có `x` điểm cùng cách tâm `i` một khoảng `d`, ta chọn `j` và `k` **khác nhau** trong `x` điểm đó:

- chọn `j` có `x` cách, rồi chọn `k` có `x - 1` cách ⇒ **`x * (x - 1)`** bộ `(j, k)` có thứ tự (đây chính là số boomerang vì thứ tự quan trọng);
- nếu không phân biệt thứ tự thì chỉ còn `x * (x - 1) / 2` cặp (tổ hợp `C(x, 2)`).

In [3]:
for x in range(0, 6):
    print(f"x={x}: có thứ tự x*(x-1)={x * (x - 1)} | không thứ tự C(x,2)={x * (x - 1) // 2} | math.perm={math.perm(x, 2)}")

x=0: có thứ tự x*(x-1)=0 | không thứ tự C(x,2)=0 | math.perm=0
x=1: có thứ tự x*(x-1)=0 | không thứ tự C(x,2)=0 | math.perm=0
x=2: có thứ tự x*(x-1)=2 | không thứ tự C(x,2)=1 | math.perm=2
x=3: có thứ tự x*(x-1)=6 | không thứ tự C(x,2)=3 | math.perm=6
x=4: có thứ tự x*(x-1)=12 | không thứ tự C(x,2)=6 | math.perm=12
x=5: có thứ tự x*(x-1)=20 | không thứ tự C(x,2)=10 | math.perm=20


### 3.3. `dict` và `Counter` đếm theo khóa

`Counter` là `dict` chuyên đếm: `cnt[d] += 1` tăng đếm cho khóa `d` (tự xem là `0` nếu chưa có). Khóa ở đây là **bình phương khoảng cách**. `cnt.values()` cho các số đếm.

In [4]:
cnt = Counter()
for d in [1, 1, 4, 1, 4, 9]:
    cnt[d] += 1
print(cnt)                    # Counter({1: 3, 4: 2, 9: 1})
print(cnt[100])               # 0: khóa chưa có
print(sum(v * (v - 1) for v in cnt.values()))     # 3*2 + 2*1 + 0 = 8

Counter({1: 3, 4: 2, 9: 1})
0
8


### 3.4. Cộng dồn khi gặp: số đếm cũ chính là số "đối tác" đã có

Thay vì đếm xong rồi mới tính `x*(x-1)`, ta có thể tính trong lúc thêm: khi gặp điểm thứ `t + 1` cùng khoảng cách (đã có `t` điểm trước đó), nó tạo thêm `t` cặp `(j, k)` với mỗi điểm trước, và mỗi cặp tính hai thứ tự ⇒ cộng `2 * t`. Tổng `Σ 2·t` với `t = 0..x-1` bằng `2 · x(x-1)/2 = x(x-1)` ✓.

In [5]:
total = 0
seen = 0
for x in range(1, 6):
    total += 2 * seen
    seen += 1
    print(f"đã có {seen} điểm: cộng dồn = {total} | x(x-1) = {seen * (seen - 1)}")

đã có 1 điểm: cộng dồn = 0 | x(x-1) = 0
đã có 2 điểm: cộng dồn = 2 | x(x-1) = 2
đã có 3 điểm: cộng dồn = 6 | x(x-1) = 6
đã có 4 điểm: cộng dồn = 12 | x(x-1) = 12
đã có 5 điểm: cộng dồn = 20 | x(x-1) = 20


### 3.5. Sắp xếp rồi đếm các đoạn bằng nhau

Một cách khác để gom: sắp xếp danh sách khoảng cách, các giá trị bằng nhau đứng cạnh nhau; đếm độ dài từng đoạn bằng nhau (bằng `itertools.groupby` hoặc vòng lặp). Tốn `O(n log n)` mỗi tâm nhưng **không cần bảng băm**.

In [6]:
ds = sorted([4, 1, 1, 9, 4, 1])
print(ds)
runs = []
i = 0
while i < len(ds):
    j = i
    while j < len(ds) and ds[j] == ds[i]:
        j += 1
    runs.append((ds[i], j - i))
    i = j
print(runs)       # [(1, 3), (4, 2), (9, 1)]

[1, 1, 1, 4, 4, 9]
[(1, 3), (4, 2), (9, 1)]


### 3.6. Khóa là bộ `(i, d)` và `defaultdict`

Một `dict` có thể có khóa là **tuple**, ví dụ `(tâm, khoảng_cách)`. `defaultdict(int)` tự khởi tạo số đếm `0` cho khóa mới: tiện cho cách "mỗi cặp một lần" ở phần sau.

In [7]:
table = defaultdict(int)
table[(0, 5)] += 1
table[(0, 5)] += 1
table[(1, 5)] += 1
print(dict(table))     # {(0, 5): 2, (1, 5): 1}

{(0, 5): 2, (1, 5): 1}


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1: `points = [[0,0],[1,0],[2,0]]`. Xét lần lượt từng điểm làm **tâm `i`** và tính bình phương khoảng cách tới hai điểm còn lại:

| Tâm `i` | Điểm khác | Bình phương khoảng cách | Nhóm bằng nhau | Số boomerang `x(x-1)` |
|---|---|---|---|---|
| `[0,0]` | `[1,0]`, `[2,0]` | `1`, `4` | `{1}:1`, `{4}:1` | `0 + 0 = 0` |
| `[1,0]` | `[0,0]`, `[2,0]` | `1`, `1` | `{1}:2` | `2 * 1 = 2` |
| `[2,0]` | `[0,0]`, `[1,0]` | `4`, `1` | `{4}:1`, `{1}:1` | `0` |

Tổng: `0 + 2 + 0 = 2` ✓.

Cách nghĩ: **cố định tâm `i`, gom các điểm khác theo khoảng cách tới `i`; mỗi nhóm `x` điểm cho `x(x-1)` boomerang.** Cộng cho mọi tâm.

In [8]:
pts = [[0, 0], [1, 0], [2, 0]]
total = 0
for i, p in enumerate(pts):
    groups = {}
    for j, q in enumerate(pts):
        if j != i:
            groups.setdefault(d2(p, q), []).append(q)
    gain = sum(len(g) * (len(g) - 1) for g in groups.values())
    print(f"tâm {p}: nhóm theo d² = { {d: g for d, g in groups.items()} } -> +{gain}")
    total += gain
print("tổng =", total)

tâm [0, 0]: nhóm theo d² = {1: [[1, 0]], 4: [[2, 0]]} -> +0
tâm [1, 0]: nhóm theo d² = {1: [[0, 0], [2, 0]]} -> +2
tâm [2, 0]: nhóm theo d² = {4: [[0, 0]], 1: [[1, 0]]} -> +0
tổng = 2


## 5. Cách 1 — Brute Force / baseline: ba vòng lặp

**Ý tưởng.** Thử mọi bộ ba có thứ tự `(i, j, k)` với `i, j, k` đôi một khác nhau; đếm khi `d²(i, j) == d²(i, k)`.

In [9]:
def boomerangs_brute(points):
    n = len(points)
    ans = 0
    for i in range(n):
        for j in range(n):
            if j == i:
                continue
            for k in range(n):
                if k == i or k == j:
                    continue
                if d2(points[i], points[j]) == d2(points[i], points[k]):
                    ans += 1
    return ans


for pts, expected in examples:
    print(boomerangs_brute(pts), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Độ phức tạp (Cách 1).** Ba vòng lặp, mỗi vòng `n` ⇒ **Time `O(n³)`**; với `n = 500` là `1.25 · 10^8` bộ ba, mỗi bộ ba còn phép tính khoảng cách — quá chậm trong Python. **Space `O(1)`**.

**Nút thắt:** với mỗi cặp `(i, j)` ta quét lại **toàn bộ** `k` chỉ để tìm các điểm cùng khoảng cách. Nếu ta **gom trước** các điểm theo khoảng cách tới `i` thì không cần quét lại.

## 6. Ý tưởng thuật toán / bước đột phá

Bộ ba `(i, j, k)` có **vai trò đặc biệt cho `i`** (tâm) và hai điểm `j`, `k` **bình đẳng** (cùng khoảng cách). Vậy:

1. Cố định tâm `i` (`n` lựa chọn).
2. Tính khoảng cách² từ `i` đến mọi điểm khác (`n - 1` phép tính) và **gom theo khoảng cách** bằng bảng băm.
3. Mỗi nhóm `x` điểm sinh `x(x - 1)` cặp `(j, k)` có thứ tự.

Tổng chi phí `O(n)` cho mỗi tâm ⇒ `O(n²)` tổng. Đó là mức tốt nhất ta có thể đạt bằng cách tổng quát vì ta phải xét mọi cặp điểm. Các cách ở mục 7 thay đổi **cách gom**: đếm rồi nhân, cộng dồn ngay, sắp xếp, hoặc xét mỗi cặp một lần.

## 7. Các cách tiếp theo

Tổng cộng **5 cách**: (1) brute force (mục 5); (2) bảng băm theo từng tâm, cuối cùng cộng `x(x-1)`; (3) bảng băm + cộng dồn khi gặp; (4) sắp xếp khoảng cách rồi đếm đoạn bằng nhau; (5) mỗi cặp điểm một lần, bảng `(tâm, khoảng cách)`. Mỗi cách: giải thích → code demo → code nộp LeetCode → độ phức tạp.

### 7.1. Cách 2 — Bảng băm cho từng tâm, rồi cộng `x(x-1)`

**Giải thích từng đoạn.**

- `for p in points:` chọn tâm `p`.
- Tạo `cnt = Counter()` **mới cho mỗi tâm** (không dùng lại giữa các tâm).
- `for q in points:` nếu `q` là chính `p` bỏ qua (so sánh theo **chỉ số** hoặc theo `q is p`; ở đây dùng điều kiện `q is not p` hoặc cứ tính khoảng cách `0` — nhóm khoảng cách `0` chỉ có một điểm nên đóng góp `0` — nhưng dùng chỉ số là an toàn nhất).
- `cnt[d2(p, q)] += 1`.
- Sau khi duyệt xong: `ans += sum(x * (x - 1) for x in cnt.values())`.

In [10]:
def boomerangs_counter(points):
    ans = 0
    for i, p in enumerate(points):
        cnt = Counter()                      # mới cho mỗi tâm
        for j, q in enumerate(points):
            if j != i:
                cnt[d2(p, q)] += 1
        ans += sum(x * (x - 1) for x in cnt.values())
    return ans


for pts, expected in examples:
    print(boomerangs_counter(pts), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


In [11]:
class SolutionCounter:
    def numberOfBoomerangs(self, points: List[List[int]]) -> int:
        ans = 0
        for i, (x1, y1) in enumerate(points):
            cnt = Counter()
            for j, (x2, y2) in enumerate(points):
                if j != i:
                    dx, dy = x1 - x2, y1 - y2
                    cnt[dx * dx + dy * dy] += 1
            ans += sum(x * (x - 1) for x in cnt.values())
        return ans


print(SolutionCounter().numberOfBoomerangs([[0, 0], [1, 0], [2, 0]]))

2


**Độ phức tạp (Cách 2).** Hai vòng lặp `n × n` ⇒ **Time `O(n²)`**; bảng đếm tối đa `n - 1` khóa ⇒ **Space `O(n)`** (được dùng lại cho mỗi tâm).

### 7.2. Cách 3 — Cộng dồn ngay khi gặp (`ans += 2 * cnt[d]`)

**Giải thích từng đoạn.** Giống Cách 2 nhưng không cần vòng `sum` cuối: trước khi tăng `cnt[d]`, số `cnt[d]` hiện tại là số điểm **đã gặp** cùng khoảng cách; điểm mới ghép với mỗi điểm đó cho `2` boomerang (hai thứ tự), nên `ans += 2 * cnt[d]` rồi mới `cnt[d] += 1`. Thứ tự "cộng trước, tăng sau" là quan trọng (đã nói ở mục 3.4).

In [12]:
def boomerangs_incremental(points):
    ans = 0
    for i, p in enumerate(points):
        cnt = {}
        for j, q in enumerate(points):
            if j == i:
                continue
            d = d2(p, q)
            c = cnt.get(d, 0)
            ans += 2 * c
            cnt[d] = c + 1
    return ans


for pts, expected in examples:
    print(boomerangs_incremental(pts), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


In [13]:
class SolutionIncremental:
    def numberOfBoomerangs(self, points: List[List[int]]) -> int:
        ans = 0
        for i, (x1, y1) in enumerate(points):
            cnt = {}
            for j, (x2, y2) in enumerate(points):
                if j == i:
                    continue
                dx, dy = x1 - x2, y1 - y2
                d = dx * dx + dy * dy
                c = cnt.get(d, 0)
                ans += 2 * c
                cnt[d] = c + 1
        return ans


print(SolutionIncremental().numberOfBoomerangs([[1, 1], [2, 2], [3, 3]]))

2


**Độ phức tạp (Cách 3).** **Time `O(n²)`**, **Space `O(n)`**; một lượt duyệt, không cần lượt `sum` cuối.

### 7.3. Cách 4 — Sắp xếp khoảng cách rồi đếm đoạn bằng nhau

**Giải thích từng đoạn.** Với mỗi tâm `i`, tạo danh sách `ds` các khoảng cách² tới mọi điểm khác, `ds.sort()`. Duyệt `ds` với con trỏ `a`; tìm `b` là cuối đoạn các giá trị bằng nhau; độ dài đoạn `x = b - a`, cộng `x * (x - 1)`; rồi `a = b`. Không dùng bảng băm — hữu ích khi muốn tránh băm hoặc khi khóa khó băm.

In [14]:
def boomerangs_sort(points):
    ans = 0
    for i, p in enumerate(points):
        ds = sorted(d2(p, q) for j, q in enumerate(points) if j != i)
        a = 0
        while a < len(ds):
            b = a
            while b < len(ds) and ds[b] == ds[a]:
                b += 1
            x = b - a
            ans += x * (x - 1)
            a = b
    return ans


for pts, expected in examples:
    print(boomerangs_sort(pts), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


In [15]:
class SolutionSort:
    def numberOfBoomerangs(self, points: List[List[int]]) -> int:
        ans = 0
        for i, (x1, y1) in enumerate(points):
            ds = []
            for j, (x2, y2) in enumerate(points):
                if j != i:
                    dx, dy = x1 - x2, y1 - y2
                    ds.append(dx * dx + dy * dy)
            ds.sort()
            a = 0
            while a < len(ds):
                b = a
                while b < len(ds) and ds[b] == ds[a]:
                    b += 1
                x = b - a
                ans += x * (x - 1)
                a = b
        return ans


print(SolutionSort().numberOfBoomerangs([[0, 0], [1, 0], [2, 0]]))

2


**Độ phức tạp (Cách 4).** Mỗi tâm sắp xếp `n - 1` số: `O(n log n)`; tổng **Time `O(n² log n)`**, **Space `O(n)`**. Chậm hơn băm một hệ số `log n` nhưng vẫn đạt với `n <= 500`.

### 7.4. Cách 5 — Mỗi cặp điểm một lần, bảng `(tâm, khoảng cách)`

**Giải thích từng đoạn.** Khoảng cách giữa `i` và `j` là **đối xứng**: `d(i, j) = d(j, i)`. Ta tính nó **một lần** cho mỗi cặp `i < j` rồi cập nhật cho **cả hai tâm**: `table[(i, d)] += 1` và `table[(j, d)] += 1`. Cuối cùng, mỗi khóa `(tâm, d)` có số đếm `x` đóng góp `x(x-1)`. Số phép tính khoảng cách giảm một nửa (`n(n-1)/2` thay vì `n(n-1)`), đổi lại bảng lớn hơn (tối đa `n(n-1)` khóa).

In [16]:
def boomerangs_pairs_once(points):
    n = len(points)
    table = defaultdict(int)
    for i in range(n):
        for j in range(i + 1, n):
            d = d2(points[i], points[j])
            table[(i, d)] += 1
            table[(j, d)] += 1
    return sum(x * (x - 1) for x in table.values())


for pts, expected in examples:
    print(boomerangs_pairs_once(pts), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


In [17]:
class SolutionPairs:
    def numberOfBoomerangs(self, points: List[List[int]]) -> int:
        n = len(points)
        table = {}
        for i in range(n):
            x1, y1 = points[i]
            for j in range(i + 1, n):
                dx, dy = x1 - points[j][0], y1 - points[j][1]
                d = dx * dx + dy * dy
                table[(i, d)] = table.get((i, d), 0) + 1
                table[(j, d)] = table.get((j, d), 0) + 1
        return sum(x * (x - 1) for x in table.values())


print(SolutionPairs().numberOfBoomerangs([[0, 0], [1, 0], [2, 0]]))

2


**Độ phức tạp (Cách 5).** **Time `O(n²)`** (nửa số phép tính khoảng cách), **Space `O(n²)`** trong trường hợp xấu (bảng có tới `n(n - 1)` khóa), nên tốn bộ nhớ hơn Cách 2/3.

## 8. TRACE từng bước

### 8.1. Bảng trạng thái — Cách 3 với `points = [[0,0],[1,0],[2,0],[1,1]]`

Tâm `i = 1` (`[1,0]`). Bình phương khoảng cách tới từng điểm khác:

| `j` | Điểm `q` | `d²` | `cnt[d]` trước | `ans += 2*cnt[d]` | `cnt` sau |
|---|---|---|---|---|---|
| 0 | `[0,0]` | 1 | 0 | +0 | `{1:1}` |
| 2 | `[2,0]` | 1 | 1 | +2 | `{1:2}` |
| 3 | `[1,1]` | 1 | 2 | +4 | `{1:3}` |

Ba điểm cùng cách `[1,0]` một khoảng `1` ⇒ `3 * 2 = 6` boomerang cho tâm này (`0 + 2 + 4 = 6` ✓).

In [18]:
def trace_incremental(points, i):
    p = points[i]
    cnt = {}
    ans = 0
    print(f"tâm i={i} p={p}")
    for j, q in enumerate(points):
        if j == i:
            continue
        d = d2(p, q)
        c = cnt.get(d, 0)
        ans += 2 * c
        cnt[d] = c + 1
        print(f"  j={j} q={q} d²={d} cnt[d] trước={c} -> ans+={2 * c} (ans={ans}) cnt={cnt}")
    return ans


pts = [[0, 0], [1, 0], [2, 0], [1, 1]]
print("tâm 1:", trace_incremental(pts, 1))
print("tâm 0:", trace_incremental(pts, 0))
print("tổng toàn bộ:", sum(trace_incremental(pts, i) for i in range(len(pts))), "| kiểm tra:", boomerangs_brute(pts))

tâm i=1 p=[1, 0]
  j=0 q=[0, 0] d²=1 cnt[d] trước=0 -> ans+=0 (ans=0) cnt={1: 1}
  j=2 q=[2, 0] d²=1 cnt[d] trước=1 -> ans+=2 (ans=2) cnt={1: 2}
  j=3 q=[1, 1] d²=1 cnt[d] trước=2 -> ans+=4 (ans=6) cnt={1: 3}
tâm 1: 6
tâm i=0 p=[0, 0]
  j=1 q=[1, 0] d²=1 cnt[d] trước=0 -> ans+=0 (ans=0) cnt={1: 1}
  j=2 q=[2, 0] d²=4 cnt[d] trước=0 -> ans+=0 (ans=0) cnt={1: 1, 4: 1}
  j=3 q=[1, 1] d²=2 cnt[d] trước=0 -> ans+=0 (ans=0) cnt={1: 1, 4: 1, 2: 1}
tâm 0: 0
tâm i=0 p=[0, 0]
  j=1 q=[1, 0] d²=1 cnt[d] trước=0 -> ans+=0 (ans=0) cnt={1: 1}
  j=2 q=[2, 0] d²=4 cnt[d] trước=0 -> ans+=0 (ans=0) cnt={1: 1, 4: 1}
  j=3 q=[1, 1] d²=2 cnt[d] trước=0 -> ans+=0 (ans=0) cnt={1: 1, 4: 1, 2: 1}
tâm i=1 p=[1, 0]
  j=0 q=[0, 0] d²=1 cnt[d] trước=0 -> ans+=0 (ans=0) cnt={1: 1}
  j=2 q=[2, 0] d²=1 cnt[d] trước=1 -> ans+=2 (ans=2) cnt={1: 2}
  j=3 q=[1, 1] d²=1 cnt[d] trước=2 -> ans+=4 (ans=6) cnt={1: 3}
tâm i=2 p=[2, 0]
  j=0 q=[0, 0] d²=4 cnt[d] trước=0 -> ans+=0 (ans=0) cnt={4: 1}
  j=1 q=[1, 0] d²=1 cnt[d] tr

### 8.2. Trace Cách 2 từng tâm với ví dụ 1

Ở mỗi tâm, in bảng đếm rồi các số hạng `x(x-1)`.

In [19]:
def trace_counter(points):
    total = 0
    for i, p in enumerate(points):
        cnt = Counter()
        for j, q in enumerate(points):
            if j != i:
                cnt[d2(p, q)] += 1
        terms = {d: x * (x - 1) for d, x in cnt.items()}
        gain = sum(terms.values())
        total += gain
        print(f"tâm i={i} p={p}: cnt={dict(cnt)} -> các số hạng x(x-1)={terms} -> +{gain}")
    return total


print(trace_counter([[0, 0], [1, 0], [2, 0]]))
print(trace_counter([[0, 0], [1, 0], [0, 1], [1, 1]]))

tâm i=0 p=[0, 0]: cnt={1: 1, 4: 1} -> các số hạng x(x-1)={1: 0, 4: 0} -> +0
tâm i=1 p=[1, 0]: cnt={1: 2} -> các số hạng x(x-1)={1: 2} -> +2
tâm i=2 p=[2, 0]: cnt={4: 1, 1: 1} -> các số hạng x(x-1)={4: 0, 1: 0} -> +0
2
tâm i=0 p=[0, 0]: cnt={1: 2, 2: 1} -> các số hạng x(x-1)={1: 2, 2: 0} -> +2
tâm i=1 p=[1, 0]: cnt={1: 2, 2: 1} -> các số hạng x(x-1)={1: 2, 2: 0} -> +2
tâm i=2 p=[0, 1]: cnt={1: 2, 2: 1} -> các số hạng x(x-1)={1: 2, 2: 0} -> +2
tâm i=3 p=[1, 1]: cnt={2: 1, 1: 2} -> các số hạng x(x-1)={2: 0, 1: 2} -> +2
8


### 8.3. Trace Cách 5 (bảng `(tâm, d)`)

Với ví dụ 1, các cặp: `(0,1): d=1`, `(0,2): d=4`, `(1,2): d=1`. Bảng: `(0,1)=1`, `(1,1)=2` (từ cặp `(0,1)` và `(1,2)`), `(0,4)=1`, `(2,4)=1`, `(2,1)=1`. Chỉ khóa `(1, 1)` có đếm `2` ⇒ `2 * 1 = 2` ✓.

In [20]:
def trace_pairs(points):
    n = len(points)
    table = defaultdict(int)
    for i in range(n):
        for j in range(i + 1, n):
            d = d2(points[i], points[j])
            table[(i, d)] += 1
            table[(j, d)] += 1
            print(f"cặp ({i},{j}) d²={d}: cập nhật ({i},{d}) và ({j},{d})")
    print("bảng:", dict(table))
    return sum(x * (x - 1) for x in table.values())


print(trace_pairs([[0, 0], [1, 0], [2, 0]]))

cặp (0,1) d²=1: cập nhật (0,1) và (1,1)
cặp (0,2) d²=4: cập nhật (0,4) và (2,4)
cặp (1,2) d²=1: cập nhật (1,1) và (2,1)
bảng: {(0, 1): 1, (1, 1): 2, (0, 4): 1, (2, 4): 1, (2, 1): 1}
2


## 9. Code nộp LeetCode

Ta nộp **Cách 3 (bảng băm theo tâm, cộng dồn ngay)**: một lượt duyệt, `O(n²)` thời gian, `O(n)` bộ nhớ.

```python
class Solution:
    def numberOfBoomerangs(self, points: List[List[int]]) -> int:
        ans = 0
        for i, (x1, y1) in enumerate(points):
            cnt = {}
            for j, (x2, y2) in enumerate(points):
                if j == i:
                    continue
                dx, dy = x1 - x2, y1 - y2
                d = dx * dx + dy * dy
                c = cnt.get(d, 0)
                ans += 2 * c
                cnt[d] = c + 1
        return ans
```

In [21]:
class Solution:
    def numberOfBoomerangs(self, points: List[List[int]]) -> int:
        ans = 0
        for i, (x1, y1) in enumerate(points):
            cnt = {}
            for j, (x2, y2) in enumerate(points):
                if j == i:
                    continue
                dx, dy = x1 - x2, y1 - y2
                d = dx * dx + dy * dy
                c = cnt.get(d, 0)
                ans += 2 * c
                cnt[d] = c + 1
        return ans


for pts, expected in examples:
    got = Solution().numberOfBoomerangs(pts)
    print(got)
    assert got == expected

2
2
0


## 10. Vì sao đúng

**Phân rã theo tâm.** Mỗi boomerang `(i, j, k)` có đúng một tâm `i`. Do đó tổng số boomerang = tổng, theo mọi `i`, của số cặp có thứ tự `(j, k)` thỏa `j != k`, `j != i`, `k != i`, `d(i, j) = d(i, k)`. Hai khoảng cách bằng nhau ⇔ bình phương bằng nhau (khoảng cách không âm), và bình phương là số nguyên nên so sánh chính xác.

**Đếm theo nhóm.** Với tâm `i` cố định, chia các điểm `!= i` thành các nhóm theo `d²`. Cặp `(j, k)` hợp lệ ⇔ `j` và `k` **cùng nhóm** và `j != k`. Nhóm `x` điểm cho `x(x - 1)` cặp có thứ tự. Các nhóm rời nhau nên tổng cộng được.

**Cộng dồn đúng.** Khi điểm thứ `t + 1` của một nhóm xuất hiện (đã có `t` điểm trước trong nhóm), nó tạo cặp với mỗi điểm trước theo hai thứ tự: thêm `2t`. Tổng `Σ_{t=0}^{x-1} 2t = x(x-1)` ✓. Vì ta **cộng trước rồi mới tăng** `cnt[d]`, giá trị `cnt[d]` đúng bằng `t`. Điểm `i` bị loại (`j == i`) để không tự ghép với mình (nhưng ngay cả khi tính nó, nhóm `d² = 0` chỉ có đúng một điểm, đóng góp `0`, vì các điểm đôi một khác nhau).

**Bảng đếm theo tâm.** Mỗi tâm dùng bảng `cnt` **mới**, vì khoảng cách chỉ so sánh được khi cùng tâm.

**Cách 4** (sắp xếp) và **Cách 5** (mỗi cặp một lần) tính cùng đại lượng: nhóm các điểm cùng khoảng cách tới một tâm — bằng đoạn bằng nhau sau sắp xếp, hoặc bằng khóa `(tâm, d)`.

## 11. Time / Space Complexity

- **Cách 2/3 (bảng băm theo tâm):** Time `O(n²)`, Space `O(n)`. Với `n = 500`: khoảng `250 000` phép tính khoảng cách.
- **Cách 4 (sắp xếp):** Time `O(n² log n)`, Space `O(n)`.
- **Cách 5 (mỗi cặp một lần):** Time `O(n²)` (nửa phép tính), Space `O(n²)`.
- **Brute force:** Time `O(n³)`, Space `O(1)`.

Không thể làm tốt hơn `O(n²)` một cách tổng quát vì kết quả phụ thuộc vào khoảng cách giữa các cặp điểm.

## 12. Bộ test

Ta gom các cách vào `impls`. **Oracle** độc lập: liệt kê mọi bộ ba có thứ tự bằng `itertools.permutations(range(n), 3)` và đếm khi bình phương khoảng cách bằng nhau (tính bằng cách khác: `sum((a - b) ** 2 ...)`).

In [22]:
def oracle(points):
    def dist2(a, b):
        return sum((u - v) ** 2 for u, v in zip(points[a], points[b]))

    return sum(1 for i, j, k in permutations(range(len(points)), 3) if dist2(i, j) == dist2(i, k))


impls = {
    "brute": boomerangs_brute,
    "counter": boomerangs_counter,
    "incremental": boomerangs_incremental,
    "sort": boomerangs_sort,
    "pairs_once": boomerangs_pairs_once,
    "leetcode": lambda p: Solution().numberOfBoomerangs(p),
    "submit_counter": lambda p: SolutionCounter().numberOfBoomerangs(p),
    "submit_incremental": lambda p: SolutionIncremental().numberOfBoomerangs(p),
    "submit_sort": lambda p: SolutionSort().numberOfBoomerangs(p),
    "submit_pairs": lambda p: SolutionPairs().numberOfBoomerangs(p),
}


def check_all(points, expected, label):
    assert oracle(points) == expected, (label, "oracle", oracle(points), expected)
    for key, fn in impls.items():
        actual = fn([p[:] for p in points])
        assert actual == expected, (label, key, points, actual, expected)
    print("PASS", label, "->", expected)


for idx, (pts, expected) in enumerate(examples, 1):
    check_all(pts, expected, f"ví dụ {idx}")

PASS ví dụ 1 -> 2
PASS ví dụ 2 -> 2
PASS ví dụ 3 -> 0


### 12.1. Các trường hợp biên

In [23]:
check_all([[0, 0]], 0, "một điểm")
check_all([[0, 0], [1, 1]], 0, "hai điểm (cần ít nhất ba)")
check_all([[0, 0], [1, 0], [0, 1]], 2, "tam giác vuông cân: đỉnh góc vuông là tâm")
check_all([[0, 0], [1, 0], [2, 0], [3, 0]], 4, "bốn điểm thẳng hàng cách đều: hai tâm giữa")
check_all([[0, 0], [1, 0], [0, 1], [1, 1]], 8, "bốn đỉnh hình vuông")
check_all([[0, 0], [2, 0], [1, 1], [1, -1]], 8, "hình thoi: tâm cách đều bốn đỉnh? (tâm hình thoi không có trong tập)")
check_all([[0, 0], [1, 0], [-1, 0], [0, 1], [0, -1]], 20, "tâm + 4 điểm cách đều (cộng thêm các tâm khác)")
check_all([[0, 0], [10000, 0], [-10000, 0]], 2, "tọa độ biên, tâm ở giữa")
check_all([[-10000, -10000], [10000, 10000], [-10000, 10000], [10000, -10000]], 8, "bốn góc của hình vuông lớn")
check_all([[0, 0], [3, 4], [4, 3], [5, 0], [0, 5]], 12, "nhiều điểm cùng cách gốc một khoảng 5")
check_all([[0, 0], [1, 0], [3, 0]], 0, "ba điểm thẳng hàng không cách đều")
check_all([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10]], 8, "năm điểm trên đường chéo, cách đều liên tiếp")

PASS một điểm -> 0
PASS hai điểm (cần ít nhất ba) -> 0
PASS tam giác vuông cân: đỉnh góc vuông là tâm -> 2
PASS bốn điểm thẳng hàng cách đều: hai tâm giữa -> 4
PASS bốn đỉnh hình vuông -> 8
PASS hình thoi: tâm cách đều bốn đỉnh? (tâm hình thoi không có trong tập) -> 8
PASS tâm + 4 điểm cách đều (cộng thêm các tâm khác) -> 20
PASS tọa độ biên, tâm ở giữa -> 2
PASS bốn góc của hình vuông lớn -> 8
PASS nhiều điểm cùng cách gốc một khoảng 5 -> 12
PASS ba điểm thẳng hàng không cách đều -> 0
PASS năm điểm trên đường chéo, cách đều liên tiếp -> 8


### 12.2. Phản ví dụ cho các cách làm sai

Bốn lỗi: (a) dùng **tổ hợp** `C(x, 2)` thay cho `x(x-1)` (quên thứ tự); (b) **dùng chung một bảng đếm** cho mọi tâm (không tạo lại); (c) khóa là **`(dx, dy)`** thay vì khoảng cách; (d) khi cộng dồn, **tăng `cnt[d]` trước rồi mới cộng** (lệch một).

In [24]:
def wrong_combination(points):
    ans = 0
    for i, p in enumerate(points):
        cnt = Counter(d2(p, q) for j, q in enumerate(points) if j != i)
        ans += sum(x * (x - 1) // 2 for x in cnt.values())     # SAI: bỏ thứ tự
    return ans


def wrong_shared_table(points):
    ans = 0
    cnt = Counter()                                            # SAI: bảng dùng chung mọi tâm
    for i, p in enumerate(points):
        for j, q in enumerate(points):
            if j != i:
                d = d2(p, q)
                ans += 2 * cnt[d]
                cnt[d] += 1
    return ans


def wrong_dxdy_key(points):
    ans = 0
    for i, p in enumerate(points):
        cnt = Counter()
        for j, q in enumerate(points):
            if j != i:
                cnt[(abs(p[0] - q[0]), abs(p[1] - q[1]))] += 1   # SAI: (1,0) và (0,1) cùng khoảng cách nhưng khác khóa
        ans += sum(x * (x - 1) for x in cnt.values())
    return ans


def wrong_increment_first(points):
    ans = 0
    for i, p in enumerate(points):
        cnt = Counter()
        for j, q in enumerate(points):
            if j != i:
                d = d2(p, q)
                cnt[d] += 1                                      # SAI: tăng trước
                ans += 2 * cnt[d]
    return ans


cases = [
    ("C(x,2) thay vì x(x-1)", wrong_combination, [[0, 0], [1, 0], [2, 0]]),
    ("bảng dùng chung", wrong_shared_table, [[0, 0], [1, 0], [2, 0]]),
    ("khóa (dx, dy)", wrong_dxdy_key, [[0, 0], [1, 0], [0, 1]]),
    ("tăng trước khi cộng", wrong_increment_first, [[0, 0], [1, 0], [2, 0]]),
]
for name, fn, pts in cases:
    print(f"{name:>24}: sai = {fn(pts)} | đúng = {Solution().numberOfBoomerangs(pts)}")
    assert fn(pts) != oracle(pts)

   C(x,2) thay vì x(x-1): sai = 1 | đúng = 2
         bảng dùng chung: sai = 14 | đúng = 2
           khóa (dx, dy): sai = 0 | đúng = 2
     tăng trước khi cộng: sai = 14 | đúng = 2


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh tập điểm **đôi một khác nhau** với tọa độ trong khoảng nhỏ (để có nhiều khoảng cách trùng nhau) cho `n <= 12`, đối chiếu mọi cách với oracle `permutations`. Thêm các lưới đều (nhiều khoảng cách trùng) và các điểm thẳng hàng.

In [25]:
def rand_points(rng, n, lo, hi):
    pts = set()
    while len(pts) < n:
        pts.add((rng.randint(lo, hi), rng.randint(lo, hi)))
    pts = [list(p) for p in pts]
    rng.shuffle(pts)
    return pts


rng = random.Random(447)
nonzero = 0
for trial in range(1500):
    span = rng.choice([2, 3, 4, 6])
    lo, hi = -span, span
    n = rng.randint(1, min(12, (hi - lo + 1) ** 2))
    pts = rand_points(rng, n, lo, hi)
    expected = oracle(pts)
    nonzero += expected > 0
    for key, fn in impls.items():
        assert fn([p[:] for p in pts]) == expected, (key, pts)
print("1500 test ngẫu nhiên OK; số ca có boomerang:", nonzero)

for trial in range(40):
    pts = rand_points(rng, rng.randint(13, 30), -10000, 10000)
    expected = boomerangs_counter(pts)
    assert Solution().numberOfBoomerangs(pts) == expected == boomerangs_sort(pts) == boomerangs_pairs_once(pts)
print("40 test tọa độ lớn: các cách O(n²) khớp nhau")

1500 test ngẫu nhiên OK; số ca có boomerang: 978
40 test tọa độ lớn: các cách O(n²) khớp nhau


### 12.3b. Tính chất: bất biến dưới phép dịch, đổi thứ tự và quay 90 độ

Số boomerang **không đổi** khi (a) dịch mọi điểm cùng một vector, (b) xáo trộn thứ tự điểm, (c) quay `90°` quanh gốc `(x, y) -> (-y, x)` (khoảng cách giữ nguyên). Đây là kiểm tra độc lập không cần oracle.

In [26]:
rng = random.Random(4470)
for trial in range(400):
    pts = rand_points(rng, rng.randint(1, 14), -5, 5)
    base = Solution().numberOfBoomerangs(pts)
    shifted = [[x + 17, y - 9] for x, y in pts]
    rotated = [[-y, x] for x, y in pts]
    shuffled = pts[:]
    rng.shuffle(shuffled)
    assert Solution().numberOfBoomerangs(shifted) == base
    assert Solution().numberOfBoomerangs(rotated) == base
    assert Solution().numberOfBoomerangs(shuffled) == base
print("bất biến dưới dịch / quay 90° / xáo trộn: OK")

bất biến dưới dịch / quay 90° / xáo trộn: OK


### 12.4. Hiệu năng với `n = 500`

Bỏ vét cạn `O(n³)`. Dùng lưới điểm (nhiều khoảng cách trùng, gần trường hợp xấu nhất về nhóm lớn) và điểm ngẫu nhiên tọa độ lớn.

In [27]:
grid = [[x, y] for x in range(-11, 11) for y in range(-12, 11)]      # 22 * 23 = 506 điểm
rng = random.Random(4471)
rng.shuffle(grid)
grid = grid[:500]
assert len(grid) == 500 and len({tuple(p) for p in grid}) == 500
big_random = rand_points(rng, 500, -10000, 10000)
fast = {k: v for k, v in impls.items() if k != "brute"}
for label, pts in [("lưới", grid), ("ngẫu nhiên lớn", big_random)]:
    results = {}
    for key, fn in fast.items():
        t0 = time.perf_counter()
        results[key] = fn([p[:] for p in pts])
        dt = time.perf_counter() - t0
        assert dt < 5, (key, dt)
    assert len(set(results.values())) == 1, results
    print(f"{label}: mọi cách đều cho {results['leetcode']} boomerang")

lưới: mọi cách đều cho 866924 boomerang


ngẫu nhiên lớn: mọi cách đều cho 4 boomerang


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng tổ hợp thay vì hoán vị (quên thứ tự)

**Code sai:** `ans += x * (x - 1) // 2`.

**Vì sao sai:** đề nói **thứ tự trong bộ ba có ý nghĩa**: `(i, j, k)` và `(i, k, j)` là hai boomerang khác nhau. Ví dụ 1 cho đáp án `2` (không phải `1`) vì hai điểm `[0,0]`, `[2,0]` đứng ở vai `j`, `k` theo hai thứ tự.

**Code đúng:** `x * (x - 1)` (hoặc cộng dồn `2 * cnt[d]`).

In [28]:
pts = [[0, 0], [1, 0], [2, 0]]
print("sai :", wrong_combination(pts))
print("đúng:", Solution().numberOfBoomerangs(pts))

sai : 1
đúng: 2


### Lỗi 2: Dùng chung một bảng đếm cho mọi tâm

**Code sai:** tạo `cnt = Counter()` **ngoài** vòng lặp tâm.

**Vì sao sai:** khoảng cách chỉ so sánh được **cùng một tâm**. Dùng chung bảng, khoảng cách `d` từ tâm này bị tính chung với khoảng cách `d` của tâm khác ⇒ đếm các bộ ba vô nghĩa (kết quả lớn hơn đúng).

**Code đúng:** tạo bảng mới ở đầu mỗi tâm.

In [29]:
pts = [[0, 0], [1, 0], [2, 0]]
print("sai :", wrong_shared_table(pts))
print("đúng:", Solution().numberOfBoomerangs(pts))

sai : 14
đúng: 2


### Lỗi 3: Dùng `(dx, dy)` làm khóa thay vì khoảng cách

**Code sai:** `cnt[(abs(dx), abs(dy))] += 1`.

**Vì sao sai:** hai điểm `(1, 0)` và `(0, 1)` cách gốc cùng một khoảng `1` nhưng có `(|dx|, |dy|)` khác nhau, nên không được gom chung ⇒ bỏ sót boomerang. Khóa phải là **số đo khoảng cách** (`dx² + dy²`).

**Code đúng:** `d = dx * dx + dy * dy`.

In [30]:
pts = [[0, 0], [1, 0], [0, 1]]
print("sai :", wrong_dxdy_key(pts))
print("đúng:", Solution().numberOfBoomerangs(pts))

sai : 0
đúng: 2


### Lỗi 4: Tăng đếm trước khi cộng dồn (lệch một)

**Code sai:** `cnt[d] += 1` rồi mới `ans += 2 * cnt[d]`.

**Vì sao sai:** lúc cộng, `cnt[d]` đã **tính cả chính điểm đang xét**, nên cộng dư `2` cho mỗi điểm: kết quả `Σ 2·(t+1)` thay vì `Σ 2·t`.

**Code đúng:** cộng `2 * cnt[d]` **trước**, rồi mới `cnt[d] += 1`.

In [31]:
pts = [[0, 0], [1, 0], [2, 0]]
print("sai :", wrong_increment_first(pts))
print("đúng:", Solution().numberOfBoomerangs(pts))

sai : 14
đúng: 2


### Lỗi 5: So sánh khoảng cách bằng số thực

**Code sai:** `cnt[math.sqrt(dx * dx + dy * dy)] += 1`, hoặc `**0.5`.

**Vì sao sai (và rủi ro):** căn bậc hai tạo số thực làm tròn; với số lớn có thể hai khoảng cách **khác nhau** cho cùng giá trị float, hoặc phép so sánh sau này `== ` lệch do sai số. Với số nguyên bình phương, mọi phép so sánh đều **chính xác**. Dưới đây ta thấy float bị làm tròn.

In [32]:
a = math.sqrt(2) * math.sqrt(2)
print(a, a == 2)                         # 2.0000000000000004 False
big = 10 ** 8
x1, x2 = big * big + 1, big * big       # hai số nguyên khác nhau
print(x1 != x2, float(x1) == float(x2))  # True True: float coi chúng bằng nhau!

2.0000000000000004 False
True True


## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận dạng:** "đếm bộ ba có điều kiện về quan hệ với một phần tử trung tâm" ⇒ **cố định phần tử trung tâm, đếm bằng bảng băm** để hạ `O(n³)` xuống `O(n²)`. Khi thứ tự có nghĩa thì dùng `x(x-1)`, không thì `C(x, 2)`.

Bảng nhắc:

| Tình huống | Gợi ý |
|---|---|
| So sánh khoảng cách Euclid | Dùng bình phương số nguyên |
| Đếm cặp có thứ tự trong nhóm `x` | `x * (x - 1)` |
| Đếm cặp không thứ tự | `x * (x - 1) // 2` |
| Bảng đếm cho từng tâm | Tạo mới trong vòng lặp tâm |
| Gom giá trị bằng nhau không dùng băm | Sắp xếp + đếm đoạn |

Bài cùng họ: **149 Max Points on a Line** (gom theo hệ số góc), **1 Two Sum**, **454 4Sum II**, **525**, **560 Subarray Sum Equals K** (đếm bằng bảng băm), **812 Largest Triangle Area**, **939 Minimum Area Rectangle**.

## 15. Biến thể

### 15.1. Chỉ đếm cặp **không** phân biệt thứ tự `(j, k)`

Số "tam giác cân tại `i` (kể cả suy biến)" không thứ tự: `Σ C(x, 2)` — đúng bằng một nửa đáp án của bài.

In [33]:
def boomerangs_unordered(points):
    ans = 0
    for i, p in enumerate(points):
        cnt = Counter(d2(p, q) for j, q in enumerate(points) if j != i)
        ans += sum(x * (x - 1) // 2 for x in cnt.values())
    return ans


rng = random.Random(4472)
for _ in range(200):
    pts = rand_points(rng, rng.randint(1, 12), -4, 4)
    assert boomerangs_unordered(pts) * 2 == Solution().numberOfBoomerangs(pts)
print(boomerangs_unordered([[0, 0], [1, 0], [2, 0]]))     # 1

1


### 15.2. Khoảng cách Manhattan (hoặc khác)

Chỉ cần thay hàm khoảng cách. Với Manhattan `|dx| + |dy|`, các điểm cùng khoảng cách tới tâm nằm trên một "hình thoi". Thuật toán gom theo khoảng cách giữ nguyên.

In [34]:
def boomerangs_with(points, dist):
    ans = 0
    for i, p in enumerate(points):
        cnt = Counter(dist(p, q) for j, q in enumerate(points) if j != i)
        ans += sum(x * (x - 1) for x in cnt.values())
    return ans


manhattan = lambda p, q: abs(p[0] - q[0]) + abs(p[1] - q[1])
chebyshev = lambda p, q: max(abs(p[0] - q[0]), abs(p[1] - q[1]))
pts = [[0, 0], [1, 0], [0, 1], [1, 1]]
print("Euclid:", boomerangs_with(pts, d2), "| Manhattan:", boomerangs_with(pts, manhattan), "| Chebyshev:", boomerangs_with(pts, chebyshev))
assert boomerangs_with(pts, d2) == Solution().numberOfBoomerangs(pts)

Euclid: 8 | Manhattan: 8 | Chebyshev: 24


### 15.3. Chỉ đếm tam giác cân **không suy biến** (loại ba điểm thẳng hàng)

Trong nhóm cùng khoảng cách tới tâm `i`, cặp `(j, k)` thẳng hàng với `i` khi `i` là **trung điểm** của `j` và `k` (vì cùng khoảng cách, hai điểm nằm khác phía). Với số cặp không thứ tự, ta trừ đi các cặp có `j + k == 2 * i` (cộng tọa độ). Bảng `seen` theo từng khoảng cách giúp tìm đối xứng của `j` qua `i` trong `O(1)`.

In [35]:
def isosceles_apex_pairs(points):
    # số cặp {j, k} không thứ tự cách đều i và i không phải trung điểm của jk
    ans = 0
    pset = {tuple(p) for p in points}
    for i, p in enumerate(points):
        cnt = Counter(d2(p, q) for j, q in enumerate(points) if j != i)
        pairs = sum(x * (x - 1) // 2 for x in cnt.values())
        # số cặp suy biến = số điểm q có điểm đối xứng 2p - q cũng thuộc tập, chia 2
        opposite = sum(1 for q in points if q != p and (2 * p[0] - q[0], 2 * p[1] - q[1]) in pset) // 2
        ans += pairs - opposite
    return ans


def isosceles_brute(points):
    n = len(points)
    ans = 0
    for i in range(n):
        for j in range(n):
            for k in range(j + 1, n):
                if i in (j, k):
                    continue
                if d2(points[i], points[j]) == d2(points[i], points[k]):
                    cross = (points[j][0] - points[i][0]) * (points[k][1] - points[i][1]) - (points[j][1] - points[i][1]) * (points[k][0] - points[i][0])
                    if cross != 0:
                        ans += 1
    return ans


rng = random.Random(4473)
for _ in range(300):
    pts = rand_points(rng, rng.randint(1, 12), -4, 4)
    assert isosceles_apex_pairs(pts) == isosceles_brute(pts), pts
print(isosceles_apex_pairs([[0, 0], [1, 0], [0, 1]]), "(tam giác vuông cân: 1 đỉnh cân)")

1 (tam giác vuông cân: 1 đỉnh cân)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ba vòng lặp | Thử mọi bộ ba có thứ tự | `O(n³)` | `O(1)` | Chỉ để hiểu đề / oracle nhỏ |
| 2. Băm theo tâm, cộng `x(x-1)` | Đếm nhóm rồi nhân | `O(n²)` | `O(n)` | Dễ hiểu, rõ ý nghĩa nhóm |
| 3. Băm theo tâm, cộng dồn | `ans += 2 * cnt[d]` rồi tăng | `O(n²)` | `O(n)` | **Lời giải nộp**: ngắn, một lượt |
| 4. Sắp xếp khoảng cách | Đếm đoạn bằng nhau | `O(n² log n)` | `O(n)` | Không muốn dùng băm |
| 5. Mỗi cặp một lần | Bảng `(tâm, d)` cập nhật hai đầu | `O(n²)` | `O(n²)` | Khi phép tính khoảng cách đắt |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `d2([0, 0], [3, 4])`, `math.perm(4, 2)`, `Solution().numberOfBoomerangs([[0, 0], [1, 1]])`, `Solution().numberOfBoomerangs([[0, 0], [1, 0], [-1, 0]])`, `Solution().numberOfBoomerangs([[0, 0], [1, 0], [2, 0], [3, 0]])`.

**Bài 2 (tính tay).** Với `points = [[0,0],[1,0],[0,1],[1,1]]` (bốn đỉnh hình vuông cạnh `1`): với mỗi tâm lập bảng khoảng cách² tới ba điểm còn lại, các nhóm và `x(x-1)`; nêu tổng.

**Bài 3 (điền TODO).** Hoàn thành `boom_todo(points)` theo Cách 2 (Counter theo tâm, cộng `x*(x-1)`).

In [36]:
def boom_todo(points):
    ans = 0
    for i, p in enumerate(points):
        # TODO 1: cnt = Counter() (mới cho mỗi tâm)
        # TODO 2: với mỗi j != i: cnt[d2(p, points[j])] += 1
        # TODO 3: ans += tổng x * (x - 1) trên cnt.values()
        pass
    return ans

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `boom_incremental2(points)` theo Cách 3 **không dùng `Counter`/`defaultdict`** (chỉ `dict` thường và `get`) và **không dùng hàm `d2`** (tính `dx*dx + dy*dy` tại chỗ).

**Bài 5 (biến thể / mini challenge).** (a) Viết `best_center(points)` trả về **chỉ số** của điểm làm tâm cho nhiều boomerang nhất (nếu hòa, chọn chỉ số nhỏ nhất), hoặc `-1` nếu `points` rỗng. (b) Viết `boomerangs_3d(points)` cho điểm trong **không gian 3 chiều** `[x, y, z]` (đổi khoảng cách² thành `dx²+dy²+dz²`) và kiểm tra rằng khi `z = 0` cho kết quả bằng bài gốc.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `25`, `12` (`4 * 3`), `0`, `2` (tâm `[0,0]`, nhóm `{1: 2}`), `4` (hai tâm giữa `[1,0]` và `[2,0]`, mỗi tâm cho `2`). Bài 3: `cnt[d2(p, points[j])] += 1` với `j != i`.
- **Mức 2 (vừa):** Bài 2: tâm `[0,0]`: `d² = 1, 1, 2` ⇒ nhóm `{1: 2}` ⇒ `2`; mỗi tâm tương tự `2` ⇒ tổng `8`. Bài 4: `c = cnt.get(d, 0); ans += 2 * c; cnt[d] = c + 1`.
- **Mức 3 (sâu):** Bài 5(a): tính `gain` của từng tâm (như Cách 2) rồi chọn `max` với `key` theo `(gain, -index)` hoặc cập nhật khi `gain > best`. 5(b): chỉ thay hàm khoảng cách; với `z = 0` mọi `dz = 0` nên khoảng cách² bằng bài 2 chiều.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [37]:
# Bài 1
assert d2([0, 0], [3, 4]) == 25
assert math.perm(4, 2) == 12
assert Solution().numberOfBoomerangs([[0, 0], [1, 1]]) == 0
assert Solution().numberOfBoomerangs([[0, 0], [1, 0], [-1, 0]]) == 2
assert Solution().numberOfBoomerangs([[0, 0], [1, 0], [2, 0], [3, 0]]) == 4

# Bài 2
assert Solution().numberOfBoomerangs([[0, 0], [1, 0], [0, 1], [1, 1]]) == 8


# Bài 3
def boom_todo_ans(points):
    ans = 0
    for i, p in enumerate(points):
        cnt = Counter()
        for j in range(len(points)):
            if j != i:
                cnt[d2(p, points[j])] += 1
        ans += sum(x * (x - 1) for x in cnt.values())
    return ans


for pts, expected in examples:
    assert boom_todo_ans(pts) == expected


# Bài 4
def boom_incremental2(points):
    ans = 0
    for i in range(len(points)):
        cnt = {}
        for j in range(len(points)):
            if j == i:
                continue
            dx = points[i][0] - points[j][0]
            dy = points[i][1] - points[j][1]
            d = dx * dx + dy * dy
            c = cnt.get(d, 0)
            ans += 2 * c
            cnt[d] = c + 1
    return ans


for pts, expected in examples:
    assert boom_incremental2(pts) == expected


# Bài 5
def best_center(points):
    best, best_idx = -1, -1
    for i, p in enumerate(points):
        cnt = Counter(d2(p, q) for j, q in enumerate(points) if j != i)
        gain = sum(x * (x - 1) for x in cnt.values())
        if gain > best:
            best, best_idx = gain, i
    return best_idx


assert best_center([]) == -1
assert best_center([[0, 0], [1, 0], [2, 0]]) == 1
assert best_center([[5, 5]]) == 0
rng = random.Random(4474)
for _ in range(200):
    pts = rand_points(rng, rng.randint(1, 10), -3, 3)
    gains = [boom_todo_ans([pts[i]] + pts[:i] + pts[i + 1:]) for i in range(len(pts))]
    assert best_center(pts) in range(len(pts))


def boomerangs_3d(points):
    ans = 0
    for i, p in enumerate(points):
        cnt = Counter()
        for j, q in enumerate(points):
            if j != i:
                cnt[sum((a - b) ** 2 for a, b in zip(p, q))] += 1
        ans += sum(x * (x - 1) for x in cnt.values())
    return ans


assert boomerangs_3d([[0, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1]]) == 12      # gốc cách đều ba điểm: 3*2 + các tâm khác
rng = random.Random(4475)
for _ in range(200):
    pts = rand_points(rng, rng.randint(1, 10), -3, 3)
    assert boomerangs_3d([p + [0] for p in pts]) == Solution().numberOfBoomerangs(pts)
print("Tất cả đáp án bài tập đều đúng.")

Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cố định tâm, đếm theo bình phương khoảng cách, cộng dồn)

```python
class Solution:
    def numberOfBoomerangs(self, points: List[List[int]]) -> int:
        ans = 0
        for i, (x1, y1) in enumerate(points):          # tâm i
            cnt = {}                                    # bảng MỚI cho mỗi tâm
            for j, (x2, y2) in enumerate(points):
                if j == i:
                    continue
                dx, dy = x1 - x2, y1 - y2
                d = dx * dx + dy * dy                   # bình phương khoảng cách (số nguyên)
                c = cnt.get(d, 0)
                ans += 2 * c                            # cộng TRƯỚC: ghép với c điểm đã gặp, 2 thứ tự
                cnt[d] = c + 1                          # rồi mới tăng
        return ans
```

**Checklist**

- [ ] Khoảng cách dùng **bình phương nguyên** (không `sqrt`, không float)?
- [ ] Bảng đếm tạo **mới cho mỗi tâm**?
- [ ] Đếm **có thứ tự**: `x * (x - 1)` (hoặc `+= 2 * cnt[d]` trước khi tăng)?
- [ ] Loại trường hợp `j == i` (hoặc biết vì sao khoảng cách `0` vô hại)?
- [ ] Đã thử `n = 1`, `n = 2`, ba điểm thẳng hàng cách đều, hình vuông (`8`), tọa độ `±10^4`?
- [ ] Skeleton đúng `def numberOfBoomerangs(self, points: List[List[int]]) -> int`?